# 🚀 Manga Web Translator - Servidor Nativo Rust (Koharu Engine na GPU)

Este notebook executa o backend nativo em **Rust + C++ (Koharu Engine)** do **Manga Web Translator** com aceleração completa por GPU (**NVIDIA T4, L4 ou A100**) no Google Colab.

### 🌟 Vantagens da Versão Rust Nativa:
- **Modelos Grandes de Alta Qualidade:** Suporte a modelos como `gemma4-12b-it` (12B), `ministral-3-8b-instruct` (8B), `gemma4-e4b-it` (4B) e muito mais na GPU do Colab!
- **Tipografia e diagramação perfeitas:** Utiliza o motor de renderização `koharu-rasterizer` e `koharu-renderer`, calculando fontes, contornos dos balões e quebras de linha automáticas para mangá.
- **Detecção e Inpainting de alta precisão:** Limpeza impecável dos balões com LaMa/AoT e detecção precisa das caixas de diálogo.
- **Mesma API do App Desktop:** A extensão do navegador se comunica nativamente com este servidor na porta 8000 via túnel seguro Cloudflare.

---
### ⚡ Passo 1: Ativar a GPU no Colab
No menu superior do Colab, vá em:  
👉 **Ambiente de Execução** ➔ **Alterar tipo de ambiente de execução** ➔ selecione **T4 GPU** (ou outra GPU disponível) e clique em **Salvar**.

In [ ]:
# Verificar a GPU conectada
!nvidia-smi

### 📦 Passo 2: Instalar Dependências do Sistema (Ubuntu), Rust e Cloudflared

In [ ]:
import os
import sys

print("📦 1/3 Instalando bibliotecas do sistema Ubuntu...")
!apt-get update -qq
!apt-get install -y -qq \
    libwebkit2gtk-4.1-dev \
    libgtk-3-dev \
    libayatana-appindicator3-dev \
    librsvg2-dev \
    libxdo-dev \
    libssl-dev \
    build-essential \
    cmake \
    pkg-config \
    libclang-dev \
    curl \
    wget \
    psmisc > /dev/null 2>&1

print("🦀 2/3 Verificando e configurando toolchain Rust...")
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ.get('PATH', '')}"
else:
    print("Rust já está instalado!")

!source $HOME/.cargo/env 2>/dev/null || true
!cargo --version

print("🌐 3/3 Instalando túnel Cloudflare (cloudflared)...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ Todas as dependências do sistema e ferramentas foram preparadas com sucesso!")

### ⚙️ Passo 3: Obter o Código e Compilar o Servidor Rust Nativo
*(Dica: Se você montar o Google Drive, o binário compilado pode ser salvo lá para iniciar em 2 segundos nas próximas sessões!)*

In [ ]:
# @title ⚙️ Obter Código e Compilar Servidor { run: "auto" }
# @markdown Marque para forçar atualização/recompilação com a versão mais recente do motor Koharu CUDA:
FORCAR_ATUALIZACAO = True # @param {type:"boolean"}

import os
import shutil
import subprocess

REPO_DIR = "/content/web-translator"
BIN_TARGET = "/content/manga-web-translator"
DRIVE_BIN = "/content/drive/MyDrive/manga-web-translator-linux"

if FORCAR_ATUALIZACAO:
    print("🔄 Forçando atualização e recompilação com o código mais recente...")
    if os.path.exists(BIN_TARGET):
        try: os.remove(BIN_TARGET)
        except Exception: pass
    if os.path.exists(DRIVE_BIN):
        try: os.remove(DRIVE_BIN)
        except Exception: pass

# 1. Verificar se temos o binário já salvo no Google Drive (inicialização instantânea)
binary_ready = False
if not FORCAR_ATUALIZACAO and os.path.exists(DRIVE_BIN):
    print(f"⚡ Encontrado binário pré-compilado no Google Drive! Copiando...")
    shutil.copy(DRIVE_BIN, BIN_TARGET)
    os.chmod(BIN_TARGET, 0o755)
    binary_ready = True
    print("✅ Binário carregado do Google Drive em 1 segundo!")

if not binary_ready:
    # 2. Clonar ou atualizar o repositório
    if not os.path.exists(REPO_DIR):
        print("📥 Clonando o repositório web-translator...")
        subprocess.run(["git", "clone", "https://github.com/alisson-porpino/web-translator.git", REPO_DIR], check=True)
    else:
        print("🔄 Atualizando repositório com o código mais recente...")
        subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=True)

    # 3. Garantir placeholders de recursos para compilação sem pacotes pré-construídos
    res_dir = f"{REPO_DIR}/desktop/src-tauri/resources"
    os.makedirs(res_dir, exist_ok=True)
    empty_zip = bytes([0x50, 0x4B, 0x05, 0x06] + [0] * 18)
    for zf in ["extension-chrome.zip", "extension-firefox.zip"]:
        zp = os.path.join(res_dir, zf)
        if not os.path.exists(zp):
            with open(zp, "wb") as f:
                f.write(empty_zip)

    # 4. Compilar o binário em modo release
    cargo_bin = os.path.expanduser("~/.cargo/bin/cargo")
    if not os.path.exists(cargo_bin):
        cargo_bin = "cargo"

    print("🔨 Compilando o servidor nativo manga-web-translator em modo release...")
    print("⏳ Aguarde enquanto o compilador otimiza o motor de inferência Koharu...")
    
    cargo_env = os.environ.copy()
    cargo_env["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{cargo_env.get('PATH', '')}"
    
    subprocess.run(
        [cargo_bin, "build", "--release", "--bin", "manga-web-translator"],
        cwd=f"{REPO_DIR}/desktop/src-tauri",
        env=cargo_env,
        check=True
    )

    COMPILED_BIN = f"{REPO_DIR}/desktop/src-tauri/target/release/manga-web-translator"
    if os.path.exists(COMPILED_BIN):
        shutil.copy(COMPILED_BIN, BIN_TARGET)
        os.chmod(BIN_TARGET, 0o755)
        print("✅ Compilação concluída com sucesso!")
        
        if os.path.exists("/content/drive/MyDrive"):
            try:
                shutil.copy(COMPILED_BIN, DRIVE_BIN)
                print("💾 Binário salvo no seu Google Drive para inicialização instantânea nas próximas vezes!")
            except Exception as e:
                print(f"Nota sobre salvar no Drive: {e}")
    else:
        raise RuntimeError("Erro: binário compilado não foi encontrado em " + COMPILED_BIN)

print(f"🚀 Binário pronto para execução: {BIN_TARGET}")

### 🧠 Passo 4: Seleção dos Modelos de IA (Tradução, OCR e Inpainting)
Aqui você pode escolher exatamente os modelos que deseja rodar na GPU. Aproveite o poder de VRAM da nuvem para usar modelos maiores e mais inteligentes!

In [ ]:
# @title ⚙️ Selecione os Modelos Desejados { run: "auto" }
# @markdown **Modelo de Tradução Neural (LLM local com raciocínio e visão):**
# @markdown - `gemma4-12b-it`: Máxima coerência e raciocínio (12B - Recomendado para T4 de 16GB, L4 e A100)
# @markdown - `ministral-3-8b-instruct`: Rápido e de altíssima precisão (8B)
# @markdown - `gemma4-e4b-it`: Excelente equilíbrio de velocidade e qualidade (4B)
# @markdown - `gemma4-e2b-it`: Modelo compacto ultra veloz (2B)
# @markdown - `lfm2.5-1.2b-instruct`: Modelo ultra leve (1.2B)
TRANSLATION_MODEL = "gemma4-12b-it" # @param ["gemma4-12b-it", "ministral-3-8b-instruct", "gemma4-e4b-it", "gemma4-e2b-it", "lfm2.5-1.2b-instruct"]

# @markdown **Quantização da LLM (Equilíbrio entre fidelidade e uso de VRAM):**
QUANTIZATION = "Q4_K_XL" # @param ["Q4_K_XL", "Q4_K_M", "Q8_0", "BF16", "F16"]

# @markdown **Modelo de Detecção e Leitura de Texto (OCR):**
# @markdown - `manga-ocr`: Especializado em mangá japonês
# @markdown - `paddleocr-vl-1.6`: Modelo versátil rápido
OCR_MODEL = "manga-ocr" # @param ["manga-ocr", "paddleocr-vl-1.6", "baberu-ocr", "hayai-ocr"]

# @markdown **Modelo de Limpeza e Reconstrução de Balões (Inpainting):**
# @markdown - `lama`: LaMa neural de altíssima fidelidade
# @markdown - `aot-inpainting`: AoT Inpainting de alta performance
INPAINTING_MODEL = "lama" # @param ["lama", "aot-inpainting", "flux2-klein", "rorem-mixed"]

print("✨ Modelos configurados para a GPU:")
print(f"  Tradução:   {TRANSLATION_MODEL} (Quantização: {QUANTIZATION})")
print(f"  OCR:        {OCR_MODEL}")
print(f"  Inpainting: {INPAINTING_MODEL}")

### 🚀 Passo 5: Iniciar o Servidor Rust Nativo e Conectar o Túnel Cloudflare

In [ ]:
import os
import time
import subprocess
import urllib.request
import json
import re

PORT = 8000
BIN_PATH = "/content/manga-web-translator"

# Defaults se a célula anterior não tiver sido executada
model = globals().get("TRANSLATION_MODEL", "gemma4-12b-it")
quant = globals().get("QUANTIZATION", "Q4_K_XL")
ocr = globals().get("OCR_MODEL", "manga-ocr")
inpainting = globals().get("INPAINTING_MODEL", "lama")

# 1. Liberar a porta 8000 e encerrar processos anteriores
print("🧹 Limpando processos anteriores...")
!fuser -k {PORT}/tcp > /dev/null 2>&1 || true
!killall -9 manga-web-translator cloudflared > /dev/null 2>&1 || true
time.sleep(1)

# 2. Configurar variáveis de ambiente do modo headless
server_env = os.environ.copy()
server_env["SERVER_MODE"] = "1"
server_env["PORT"] = str(PORT)
server_env["RUST_LOG"] = "info"
server_env["KOHARU_TRANSLATION_MODEL"] = model
server_env["KOHARU_QUANTIZATION"] = quant
server_env["KOHARU_OCR"] = ocr
server_env["KOHARU_INPAINTING"] = inpainting

# 3. Iniciar o servidor Rust nativo em background com os modelos configurados
cmd = [
    BIN_PATH,
    "--server",
    "--port", str(PORT),
    "--model", model,
    "--quantization", quant,
    "--ocr", ocr,
    "--inpainting", inpainting
]
print(f"🚀 Iniciando servidor nativo Koharu na porta {PORT}...")
print(f"   Model: {model} | OCR: {ocr} | Inpainting: {inpainting}")
server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    cmd,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    env=server_env
)

# 4. Aguardar o servidor responder no healthcheck
print("⏳ Aguardando inicialização do servidor...")
ready = False
for i in range(45):
    time.sleep(1)
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/healthz")
        with urllib.request.urlopen(req, timeout=1) as resp:
            data = json.loads(resp.read().decode())
            if data.get("ok"):
                ready = True
                break
    except Exception:
        pass

if not ready:
    print("⚠️ O servidor demorou para responder. Verifique os logs:")
    !tail -n 25 /content/server.log
else:
    print(f"✅ Servidor Rust nativo ativo e respondendo na porta {PORT}!")

# 5. Iniciar o túnel Cloudflare
print("🌐 Conectando túnel seguro da Cloudflare...")
tunnel_log = open("/content/tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "=" * 65)
    print("🎉 SEU SERVIDOR NATIVO RUST ESTÁ ONLINE COM SUCESSO!")
    print(f"👉 URL DO SERVIDOR: {public_url}")
    print("=" * 65)
    print("\n📋 Próximos passos:")
    print("1. Abra o popup da extensão Manga Web Translator no seu navegador.")
    print("2. Clique no ícone de engrenagem ⚙️ (Configurações).")
    print(f"3. No campo 'URL do servidor desktop', cole:")
    print(f"   {public_url}")
    print("4. Clique em Salvar.")
    print("5. O status ficará verde 🟢 Conectado ao aplicativo desktop.")
    print("6. Abra qualquer capítulo de mangá no Rawkuma e aproveite a qualidade máxima!\n")
else:
    print("❌ Não foi possível obter o link do Cloudflare. Verifique /content/tunnel.log.")

# 6. Monitoramento de logs em tempo real
print("📊 Logs do servidor em tempo real (Pressione o botão de parar no Colab para encerrar):")
try:
    !tail -f /content/server.log
except KeyboardInterrupt:
    print("\n🛑 Encerrando servidor e túnel...")
    server_proc.terminate()
    tunnel_proc.terminate()
